# Week 4: transfer learning when labels are scarce

Labelling satellite tiles costs expert time, so here we assume only **100 labelled tiles per class**
(1,000 total, under 4% of EuroSAT) and compare:

| | model | what is trained |
|---|---|---|
| A | SatNet from scratch (Week 3 architecture) | everything |
| B | ImageNet ResNet-18, frozen (feature extraction) | only the new head |
| C | ResNet-18, fine-tuned | last block + head, with augmentation |

Then: how many labels does transfer learning need, does normalisation matter, how to handle a rare
class, and **where the model looks** (Grad-CAM).

The test set is the same 4,050 tiles as in Week 3 (same seed, same split).

In [ ]:
# Setup: works in Google Colab and locally (from the repo root or from this folder)
import os, sys, subprocess

REPO_URL = "https://github.com/abdulloh006/satellite-scene-recognition.git"  # <- your repo
REPO_DIR = "satellite-scene-recognition"

if "google.colab" in sys.modules:
    if not os.path.exists(REPO_DIR) and not os.getcwd().endswith(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    if os.path.exists(REPO_DIR):
        os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()).startswith("week"):
    os.chdir("..")  # opened from week3_cnn/ or week4_transfer/: go to the repo root

sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import WeightedRandomSampler
from sklearn.linear_model import LogisticRegression

from scenelab import data as D, engine as E, metrics as M, models as MD, plots as P

SEED = 42
E.seed_everything(SEED)
device = E.get_device()
SMOKE = os.environ.get("SMOKE_TEST") == "1"   # tiny run, only used to test the pipeline
GPU = device.type == "cuda"

N_TRAIN, N_VAL = 100, 50                         # labelled images per class
IMG = 112 if GPU and not SMOKE else 64           # ResNet input size (tiles are 64x64)
EP_SCRATCH, EP_HEAD, EP_FT, EP_IMB = (1, 1, 1, 1) if SMOKE else ((15, 6, 8, 5) if GPU else (5, 3, 3, 2))
WORKERS = 0 if SMOKE else 2
print(f"device {device} | input {IMG}px | epochs scratch/head/fine-tune/imbalance: "
      f"{EP_SCRATCH}/{EP_HEAD}/{EP_FT}/{EP_IMB}")

## 1. Data: a small labelled training set, the full Week 3 test set

In [ ]:
base = D.load_eurosat("./data")
CLASSES = D.CLASSES
targets = np.asarray(base.targets)
train_pool, val_pool, test_idx = D.stratified_split(targets, (0.70, 0.15, 0.15), seed=SEED)

train_idx = D.per_class_subset(train_pool, targets, N_TRAIN, SEED)
val_idx = D.per_class_subset(val_pool, targets, N_VAL, SEED)
assert not (set(train_idx) | set(val_idx)) & set(test_idx), "LEAKAGE"
print(f"train {len(train_idx)} | val {len(val_idx)} | test {len(test_idx)}")
print("train per class:", D.class_counts(train_idx, targets))

# Scratch model: EuroSAT statistics. Pretrained model: the ImageNet statistics it was trained with.
SAT_MEAN, SAT_STD = D.compute_mean_std(base, train_pool, max_images=500 if SMOKE else 4000, seed=SEED)
IN_MEAN, IN_STD = D.IMAGENET_MEAN, D.IMAGENET_STD

def loaders(mean, std, size, train_indices=train_idx, augment=True, sampler=None, bs=64):
    tr = D.make_loader(base, train_indices, D.make_transforms(mean, std, size, augment=augment),
                       bs, shuffle=True, num_workers=WORKERS, sampler=sampler)
    ev = D.make_transforms(mean, std, size)
    va = D.make_loader(base, val_idx, ev, 256, num_workers=WORKERS)
    te = D.make_loader(base, test_idx, ev, 256, num_workers=WORKERS)
    return tr, va, te

sat_train, sat_val, sat_test = loaders(SAT_MEAN, SAT_STD, None)
in_train_plain, in_val, in_test = loaders(IN_MEAN, IN_STD, IMG, augment=False)
in_train_aug, _, _ = loaders(IN_MEAN, IN_STD, IMG, augment=True)

### Reference: the Week 3 model trained on all 18,900 training tiles

In [ ]:
week3_acc = None
if os.path.exists("checkpoints/satnet_week3.pt"):
    ck = torch.load("checkpoints/satnet_week3.pt", map_location=device)
    w3 = MD.SatNet().to(device)
    w3.load_state_dict(ck["state_dict"])
    w3_test = D.make_loader(base, test_idx, D.make_transforms(ck["mean"], ck["std"]), 256, num_workers=WORKERS)
    _, week3_acc = E.run_epoch(w3, w3_test, device, nn.CrossEntropyLoss())
    print(f"Week 3 SatNet (18,900 labels) test accuracy: {week3_acc:.1%}")
else:
    print("checkpoints/satnet_week3.pt not found: run the Week 3 notebook first to get this reference.")

## A. SatNet from scratch on 1,000 labels

In [ ]:
E.seed_everything(SEED)
scratch = MD.SatNet().to(device)
opt = torch.optim.AdamW(scratch.parameters(), lr=3e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, epochs=EP_SCRATCH, steps_per_epoch=len(sat_train))
hist_scratch = E.fit(scratch, sat_train, sat_val, device, EP_SCRATCH, opt, scheduler=sched, tag="scratch")

## B. Feature extraction: frozen ImageNet ResNet-18, new linear head

In [ ]:
E.seed_everything(SEED)
fe_model = MD.build_resnet18(num_classes=10, pretrained=True).to(device)
total = MD.count_params(fe_model, trainable_only=False)
print(f"total parameters {total:,} | trainable {MD.count_params(fe_model):,} "
      f"({MD.count_params(fe_model) / total:.2%})")
opt = torch.optim.AdamW([p for p in fe_model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-2)
hist_fe = E.fit(fe_model, in_train_plain, in_val, device, EP_HEAD, opt, tag="feature-extraction")

### How many labels does a pretrained backbone need?

Run the frozen backbone once to turn each tile into a 512-number embedding, then fit a logistic
regression on 5, 10, 25, 50 and 100 labelled tiles per class. Because the backbone is frozen, this is
the same model as B but takes seconds per point.

In [ ]:
backbone = MD.build_resnet18(pretrained=True).to(device)
backbone.fc = nn.Identity()
F_pool, y_pool = E.extract_features(backbone, in_train_plain, device)   # the 100-per-class pool
F_test, y_test = E.extract_features(backbone, in_test, device)
print("embeddings:", F_pool.shape, F_test.shape)

curve = {}
for n in [5, 10, 25, 50, 100]:
    pick = np.concatenate([np.flatnonzero(y_pool == c)[:n] for c in range(10)])
    clf = LogisticRegression(max_iter=3000, C=1.0).fit(F_pool[pick], y_pool[pick])
    curve[n] = clf.score(F_test, y_test)
    print(f"{n:>3} labels/class -> test accuracy {curve[n]:.1%}")

plt.figure(figsize=(6, 3.5))
plt.plot(list(curve), list(curve.values()), "o-", label="ResNet-18 embeddings + logistic regression")
if week3_acc is not None:
    plt.axhline(week3_acc, ls="--", c="gray", label="Week 3 SatNet, 1,890 labels/class")
plt.xscale("log"); plt.xticks(list(curve), list(curve)); plt.ylim(0, 1)
plt.xlabel("labelled tiles per class"); plt.ylabel("test accuracy"); plt.legend(fontsize=8)
plt.title("Data efficiency of transfer learning"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## C. Fine-tuning with satellite-friendly augmentation

Satellite tiles have no "up", so vertical flips and 90-degree rotations are valid, which is not
true for ordinary photos.

In [ ]:
aug_tf = D.make_transforms(IN_MEAN, IN_STD, IMG, augment=True)
img, label = base[train_idx[3]]
fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
axes[0].imshow(img.resize((IMG, IMG))); axes[0].set_title("original: " + CLASSES[label], fontsize=8)
for ax in axes[1:]:
    ax.imshow(D.denormalize(aug_tf(img), IN_MEAN, IN_STD)); ax.set_title("augmented", fontsize=8)
for ax in axes: ax.axis("off")
plt.show()

In [ ]:
ft_model = copy.deepcopy(fe_model)       # start from the trained head, so the new layer is not random
MD.unfreeze(ft_model.layer4)
ft_model.fc[0].p = 0.3                   # 30% dropout before the classifier

opt = torch.optim.AdamW([
    {"params": ft_model.layer4.parameters(), "lr": 1e-4},   # pretrained: small steps
    {"params": ft_model.fc.parameters(), "lr": 1e-3},       # new head: larger steps
], weight_decay=1e-2)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EP_FT * len(in_train_aug))
print(f"trainable parameters: {MD.count_params(ft_model):,} of {MD.count_params(ft_model, False):,}")
hist_ft = E.fit(ft_model, in_train_aug, in_val, device, EP_FT, opt, scheduler=sched, tag="fine-tune")

## Comparison

In [ ]:
P.plot_histories({"A. Scratch SatNet": hist_scratch, "B. Feature extraction": hist_fe,
                  "C. Fine-tuned ResNet-18": hist_ft})

rows = [("A. Scratch SatNet", scratch, sat_test, hist_scratch),
        ("B. Feature extraction", fe_model, in_test, hist_fe),
        ("C. Fine-tuned ResNet-18", ft_model, in_test, hist_ft)]
print(f"{'model':<26}{'trainable params':>18}{'best val':>10}{'test':>8}")
test_acc = {}
for name, m, loader, h in rows:
    _, test_acc[name] = E.run_epoch(m, loader, device, nn.CrossEntropyLoss())
    print(f"{name:<26}{MD.count_params(m):>18,}{max(h['val_acc']):>10.1%}{test_acc[name]:>8.1%}")
if week3_acc is not None:
    print(f"{'Week 3 SatNet (all data)':<26}{MD.count_params(w3):>18,}{'':>10}{week3_acc:>8.1%}")

In [ ]:
probs, ys = E.predict(ft_model, in_test, device)
preds = probs.argmax(1)
probs_s, ys_s = E.predict(scratch, sat_test, device)
rec_scratch = M.recall_per_class(ys_s, probs_s.argmax(1), 10)

M.per_class_table(ys, preds, CLASSES, extra_recall=rec_scratch, extra_name="scratch")
cm = M.confusion(ys, preds, 10)
P.plot_confusion(cm, CLASSES, title="Fine-tuned ResNet-18")
print("Most common mistakes of the fine-tuned model:")
for t, p, n in M.top_confusions(cm, CLASSES):
    print(f"  {t} predicted as {p}: {n} times")

### Scene-level view: missed water is the costly error

For a flood-mapping or water-monitoring use, a water tile labelled as land is worse than
mixing up two crop types.

In [ ]:
g_acc, g_cm = M.group_accuracy(ys, preds, CLASSES, D.GROUPS)
groups = list(D.GROUPS)
w = groups.index("water")
print(f"10-class accuracy : {(preds == ys).float().mean():.2%}")
print(f"4-group accuracy  : {g_acc:.2%}")
print(f"water recall      : {g_cm[w, w] / g_cm[w].sum():.1%}  "
      f"({g_cm[w].sum() - g_cm[w, w]} water tiles called land)")
P.plot_confusion(g_cm, groups, title="Group-level confusion (fine-tuned)")

## Does normalisation matter for a pretrained model?

In [ ]:
variants = {
    "ImageNet stats (what it was trained with)": D.make_transforms(IN_MEAN, IN_STD, IMG),
    "EuroSAT stats": D.make_transforms(SAT_MEAN, SAT_STD, IMG),
    "no normalisation (0..1)": D.make_transforms((0, 0, 0), (1, 1, 1), IMG),
}
for name, tf in variants.items():
    loader = D.make_loader(base, val_idx, tf, 256, num_workers=WORKERS)
    _, acc = E.run_epoch(ft_model, loader, device, nn.CrossEntropyLoss())
    print(f"{name:<45} val accuracy {acc:.1%}")

## A rare class: River with only 10 labels

Three fixes compared, each starting from a fresh pretrained ResNet-18 with `layer4` unfrozen:
1. plain cross-entropy (do nothing)
2. class-weighted cross-entropy (mistakes on River cost more)
3. `WeightedRandomSampler` (show River tiles more often)

In [ ]:
RIVER = CLASSES.index("River")
imb_idx = [i for i in train_idx if targets[i] != RIVER] + \
          [i for i in train_idx if targets[i] == RIVER][:10]
counts = D.class_counts(imb_idx, targets)
print("training counts:", counts)

class_w = torch.tensor(counts.sum() / (10 * counts), dtype=torch.float32, device=device)
sample_w = (1.0 / counts)[targets[imb_idx]]
sampler = WeightedRandomSampler(torch.as_tensor(sample_w, dtype=torch.double), len(imb_idx), replacement=True)

def train_imbalanced(criterion, sampler=None, tag=""):
    E.seed_everything(SEED)
    m = MD.build_resnet18(pretrained=True).to(device)
    MD.unfreeze(m.layer4)
    o = torch.optim.AdamW([{"params": m.layer4.parameters(), "lr": 1e-4},
                           {"params": m.fc.parameters(), "lr": 1e-3}], weight_decay=1e-2)
    tr, va, _ = loaders(IN_MEAN, IN_STD, IMG, train_indices=imb_idx, sampler=sampler)
    E.fit(m, tr, va, device, EP_IMB, o, criterion, tag=tag)
    pr, y = E.predict(m, va, device)
    p = pr.argmax(1)
    river_recall = (p[y == RIVER] == RIVER).float().mean().item()
    macro = M.recall_per_class(y, p, 10).mean()
    return river_recall, macro

imb = {
    "plain CE": train_imbalanced(nn.CrossEntropyLoss(), tag="plain"),
    "weighted CE": train_imbalanced(nn.CrossEntropyLoss(weight=class_w), tag="weighted"),
    "weighted sampler": train_imbalanced(nn.CrossEntropyLoss(), sampler=sampler, tag="sampler"),
}
print(f"\n{'method':<18}{'River recall':>14}{'balanced acc':>14}")
for k, (rr, ba) in imb.items():
    print(f"{k:<18}{rr:>14.1%}{ba:>14.1%}")

## Where does the model look? Grad-CAM

Red = regions whose features increased the score of the predicted class (last ResNet block).

In [ ]:
cam = MD.GradCAM(ft_model, ft_model.layer4)
rng = np.random.default_rng(SEED)
show = rng.choice(len(test_idx), size=6, replace=False)
ev = D.make_transforms(IN_MEAN, IN_STD, IMG)
x = torch.stack([ev(base[test_idx[i]][0]) for i in show]).to(device)
heat, pred = cam(x)
cam.remove()

fig, axes = plt.subplots(2, 6, figsize=(15, 5.4))
for j, i in enumerate(show):
    img = D.denormalize(x[j], IN_MEAN, IN_STD)
    true = base.targets[test_idx[i]]
    axes[0, j].imshow(img); axes[0, j].set_title(f"true: {CLASSES[true]}", fontsize=8)
    axes[1, j].imshow(img); axes[1, j].imshow(heat[j], cmap="jet", alpha=0.45)
    axes[1, j].set_title(f"pred: {CLASSES[pred[j]]}", fontsize=8)
for ax in axes.flat: ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
os.makedirs("checkpoints", exist_ok=True)
torch.save({"state_dict": ft_model.state_dict(), "img_size": IMG, "classes": CLASSES},
           "checkpoints/resnet18_finetuned_week4.pt")
print("saved checkpoints/resnet18_finetuned_week4.pt")

## Observations (write these yourself after running)

* With 100 labels/class, how far is each of A, B, C from the Week 3 model trained on ~19x more labels?
* From the data-efficiency curve: how many labels per class does the frozen backbone need to beat the HOG + SVM baseline from Week 2?
* Which classes gained most from transfer learning, and why might ImageNet features help them?
* Which fix for the rare River class worked best, and what did it cost the other classes?
* Do the Grad-CAM maps look at sensible things (the river line, road edges), or at shortcuts?